# 📊 Exploratory Data Analysis — Cats vs Dogs Dataset

**PRODIGY_ML_03** | Prodigy InfoTech — ML Internship Task 03

This notebook explores the Kaggle Dogs vs Cats dataset to understand:
- Dataset structure and class distribution
- Image dimensions and properties
- Sample visualizations
- HOG feature extraction visualization
- Pixel intensity distributions

**Dataset:** [Kaggle Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats/data)

## 1. Import Libraries

In [ ]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.feature import hog
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.style.use('dark_background')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries imported successfully!')

: 

## 2. Load and Explore Dataset

In [ ]:
# Path to the training dataset
# Update this path to where your dataset is stored
DATASET_PATH = os.path.join('..', '..', '..', 'artifacts', 'raw_data', 'train')

# If the organized structure exists, use it
cats_path = os.path.join(DATASET_PATH, 'cats')
dogs_path = os.path.join(DATASET_PATH, 'dogs')

if os.path.exists(cats_path):
    cat_images = [os.path.join(cats_path, f) for f in os.listdir(cats_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    dog_images = [os.path.join(dogs_path, f) for f in os.listdir(dogs_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
else:
    # Fallback: If images are in a flat directory
    all_files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    cat_images = [os.path.join(DATASET_PATH, f) for f in all_files if f.lower().startswith('cat')]
    dog_images = [os.path.join(DATASET_PATH, f) for f in all_files if f.lower().startswith('dog')]

print(f'Total Cat images: {len(cat_images)}')
print(f'Total Dog images: {len(dog_images)}')
print(f'Total images: {len(cat_images) + len(dog_images)}')

## 3. Class Distribution

In [ ]:
# Class distribution
classes = ['Cats 🐱', 'Dogs 🐶']
counts = [len(cat_images), len(dog_images)]
colors = ['#8b5cf6', '#3b82f6']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(classes, counts, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_title('Class Distribution', fontsize=16, fontweight='bold', color='#a78bfa')
axes[0].set_ylabel('Number of Images', fontsize=12)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{count:,}', ha='center', fontsize=14, fontweight='bold', color='white')

# Pie chart
axes[1].pie(counts, labels=classes, autopct='%1.1f%%', colors=colors,
            startangle=90, textprops={'fontsize': 13, 'color': 'white'},
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Proportion', fontsize=16, fontweight='bold', color='#a78bfa')

plt.tight_layout()
plt.show()

print(f'\nClass Balance Ratio: {min(counts)/max(counts):.4f} (1.0 = perfectly balanced)')

## 4. Sample Image Visualization

In [ ]:
# Display sample images
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Sample Images from Dataset', fontsize=18, fontweight='bold', color='#a78bfa', y=1.02)

# Sample cats
for i, img_path in enumerate(cat_images[:5]):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Cat #{i+1}', fontsize=11, color='#c4b5fd')
    axes[0, i].axis('off')

# Sample dogs
for i, img_path in enumerate(dog_images[:5]):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'Dog #{i+1}', fontsize=11, color='#93c5fd')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('CATS', fontsize=14, fontweight='bold', color='#8b5cf6')
axes[1, 0].set_ylabel('DOGS', fontsize=14, fontweight='bold', color='#3b82f6')

plt.tight_layout()
plt.show()

## 5. Image Dimension Analysis

In [ ]:
# Analyze image dimensions (sample 500 images from each class)
sample_size = min(500, len(cat_images), len(dog_images))

widths_cats, heights_cats = [], []
widths_dogs, heights_dogs = [], []

for img_path in cat_images[:sample_size]:
    img = cv2.imread(img_path)
    if img is not None:
        h, w = img.shape[:2]
        widths_cats.append(w)
        heights_cats.append(h)

for img_path in dog_images[:sample_size]:
    img = cv2.imread(img_path)
    if img is not None:
        h, w = img.shape[:2]
        widths_dogs.append(w)
        heights_dogs.append(h)

# Create DataFrame for analysis
dims_df = pd.DataFrame({
    'Width': widths_cats + widths_dogs,
    'Height': heights_cats + heights_dogs,
    'Class': ['Cat'] * len(widths_cats) + ['Dog'] * len(widths_dogs)
})

print('Image Dimension Statistics:')
print('=' * 50)
print(dims_df.groupby('Class')[['Width', 'Height']].describe().round(1))

# Scatter plot of dimensions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(widths_cats, heights_cats, alpha=0.3, s=10, c='#8b5cf6', label='Cats')
axes[0].scatter(widths_dogs, heights_dogs, alpha=0.3, s=10, c='#3b82f6', label='Dogs')
axes[0].set_xlabel('Width (pixels)', fontsize=12)
axes[0].set_ylabel('Height (pixels)', fontsize=12)
axes[0].set_title('Image Dimensions Distribution', fontsize=14, fontweight='bold', color='#a78bfa')
axes[0].legend(fontsize=11)

# Aspect ratio distribution
aspect_cats = [w/h for w, h in zip(widths_cats, heights_cats)]
aspect_dogs = [w/h for w, h in zip(widths_dogs, heights_dogs)]

axes[1].hist(aspect_cats, bins=30, alpha=0.6, color='#8b5cf6', label='Cats', edgecolor='white')
axes[1].hist(aspect_dogs, bins=30, alpha=0.6, color='#3b82f6', label='Dogs', edgecolor='white')
axes[1].set_xlabel('Aspect Ratio (W/H)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Aspect Ratio Distribution', fontsize=14, fontweight='bold', color='#a78bfa')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

## 6. Pixel Intensity Analysis

In [ ]:
# Analyze pixel intensity distributions
sample_n = min(100, len(cat_images), len(dog_images))

cat_means, dog_means = [], []
cat_stds, dog_stds = [], []

for img_path in cat_images[:sample_n]:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img_resized = cv2.resize(img, (64, 64))
        cat_means.append(np.mean(img_resized))
        cat_stds.append(np.std(img_resized))

for img_path in dog_images[:sample_n]:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img_resized = cv2.resize(img, (64, 64))
        dog_means.append(np.mean(img_resized))
        dog_stds.append(np.std(img_resized))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean intensity
axes[0].hist(cat_means, bins=25, alpha=0.6, color='#8b5cf6', label='Cats', edgecolor='white')
axes[0].hist(dog_means, bins=25, alpha=0.6, color='#3b82f6', label='Dogs', edgecolor='white')
axes[0].set_title('Mean Pixel Intensity', fontsize=14, fontweight='bold', color='#a78bfa')
axes[0].set_xlabel('Mean Intensity', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].legend(fontsize=11)

# Std intensity
axes[1].hist(cat_stds, bins=25, alpha=0.6, color='#8b5cf6', label='Cats', edgecolor='white')
axes[1].hist(dog_stds, bins=25, alpha=0.6, color='#3b82f6', label='Dogs', edgecolor='white')
axes[1].set_title('Pixel Intensity Std Dev', fontsize=14, fontweight='bold', color='#a78bfa')
axes[1].set_xlabel('Std Deviation', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'Cat images — Mean Intensity: {np.mean(cat_means):.2f} ± {np.std(cat_means):.2f}')
print(f'Dog images — Mean Intensity: {np.mean(dog_means):.2f} ± {np.std(dog_means):.2f}')

## 7. HOG Feature Visualization

In [ ]:
# Visualize HOG features for sample images
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('HOG Feature Extraction Visualization', fontsize=18, fontweight='bold', color='#a78bfa', y=1.02)

TARGET_SIZE = (64, 64)

for idx, (img_path, label, color) in enumerate([
    (cat_images[0], 'Cat', '#c4b5fd'),
    (dog_images[0], 'Dog', '#93c5fd')
]):
    # Read and preprocess
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, TARGET_SIZE)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
    
    # Extract HOG with visualization
    hog_features, hog_image = hog(
        img_gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=True,
        feature_vector=True
    )
    
    row = idx
    
    # Original image
    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f'{label} — Original', fontsize=11, color=color)
    axes[row, 0].axis('off')
    
    # Resized image
    axes[row, 1].imshow(img_resized)
    axes[row, 1].set_title(f'{label} — Resized (64×64)', fontsize=11, color=color)
    axes[row, 1].axis('off')
    
    # Grayscale
    axes[row, 2].imshow(img_gray, cmap='gray')
    axes[row, 2].set_title(f'{label} — Grayscale', fontsize=11, color=color)
    axes[row, 2].axis('off')
    
    # HOG visualization
    axes[row, 3].imshow(hog_image, cmap='magma')
    axes[row, 3].set_title(f'{label} — HOG Features', fontsize=11, color=color)
    axes[row, 3].axis('off')

plt.tight_layout()
plt.show()

print(f'HOG feature vector length: {len(hog_features)}')
print(f'HOG parameters: orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2)')

## 8. HOG Feature Distribution

In [ ]:
# Compare HOG feature distributions between cats and dogs
n_samples = min(50, len(cat_images), len(dog_images))
cat_hog_features = []
dog_hog_features = []

for img_path in cat_images[:n_samples]:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, TARGET_SIZE)
        features = hog(img, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys', feature_vector=True)
        cat_hog_features.append(features)

for img_path in dog_images[:n_samples]:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, TARGET_SIZE)
        features = hog(img, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys', feature_vector=True)
        dog_hog_features.append(features)

cat_hog_mean = np.mean(cat_hog_features, axis=0)
dog_hog_mean = np.mean(dog_hog_features, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Mean HOG feature comparison
axes[0].plot(cat_hog_mean[:100], color='#8b5cf6', alpha=0.8, label='Cats', linewidth=1.5)
axes[0].plot(dog_hog_mean[:100], color='#3b82f6', alpha=0.8, label='Dogs', linewidth=1.5)
axes[0].set_title('Mean HOG Features (first 100)', fontsize=14, fontweight='bold', color='#a78bfa')
axes[0].set_xlabel('Feature Index', fontsize=12)
axes[0].set_ylabel('Feature Value', fontsize=12)
axes[0].legend(fontsize=11)

# Feature difference
diff = np.abs(cat_hog_mean - dog_hog_mean)
axes[1].bar(range(50), diff[:50], color='#a78bfa', alpha=0.8, edgecolor='white', linewidth=0.5)
axes[1].set_title('Absolute Feature Difference (first 50)', fontsize=14, fontweight='bold', color='#a78bfa')
axes[1].set_xlabel('Feature Index', fontsize=12)
axes[1].set_ylabel('|Cat - Dog| Difference', fontsize=12)

plt.tight_layout()
plt.show()

print(f'\nMost discriminative features (highest difference): {np.argsort(diff)[-5:]}')

## 9. Color Channel Analysis

In [ ]:
# Analyze color channels
sample_cat = cv2.imread(cat_images[0])
sample_cat = cv2.resize(sample_cat, (128, 128))
sample_dog = cv2.imread(dog_images[0])
sample_dog = cv2.resize(sample_dog, (128, 128))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Color Channel Decomposition', fontsize=18, fontweight='bold', color='#a78bfa', y=1.02)

channel_names = ['Blue', 'Green', 'Red']
channel_cmaps = ['Blues', 'Greens', 'Reds']

for idx, (img, label, title_color) in enumerate([
    (sample_cat, 'Cat', '#c4b5fd'),
    (sample_dog, 'Dog', '#93c5fd')
]):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[idx, 0].imshow(img_rgb)
    axes[idx, 0].set_title(f'{label} — Original', fontsize=11, color=title_color)
    axes[idx, 0].axis('off')
    
    for c in range(3):
        axes[idx, c+1].imshow(img[:, :, c], cmap=channel_cmaps[c])
        axes[idx, c+1].set_title(f'{label} — {channel_names[c]}', fontsize=11, color=title_color)
        axes[idx, c+1].axis('off')

plt.tight_layout()
plt.show()

## 10. Key Findings Summary

### Observations:
1. **Balanced Dataset** — The dataset has roughly equal number of cat and dog images
2. **Variable Dimensions** — Images have varying dimensions, justifying the resize step to 64×64
3. **HOG Features** — HOG captures edge and gradient patterns effectively, with visible differences between cat and dog feature distributions
4. **Pixel Intensity** — Cat and dog images have overlapping but slightly different intensity distributions
5. **Feature Dimensionality** — HOG produces a compact feature vector suitable for SVM classification

### Preprocessing Pipeline Decision:
- **Resize** to 64×64 (standardize input dimensions)
- **Grayscale** conversion (reduce complexity, HOG works on gradients)
- **HOG feature extraction** (capture edge patterns)
- **StandardScaler** normalization (SVM performs better with scaled features)